<a href="https://colab.research.google.com/github/put-star/Flask_Blog1/blob/master/project_ecommers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

folder_path = '/content/drive/MyDrive/data ecommers'

# Tampilkan semua file di folder
for file in os.listdir(folder_path):
    print(file)

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# 1. LOAD DATA
# ============================================================
df = pd.read_csv('/content/drive/MyDrive/data economic GDP/final_global_economic_crisis_dataset.csv')
print(f"Shape awal: {df.shape}")

# ============================================================
# 2. INSPEKSI AWAL
# ============================================================
print("\n--- Info ---")
print(df.info())
print("\n--- Missing values ---")
print(df.isnull().sum())
print("\n--- Duplikat:", df.duplicated().sum())
print("\n--- Statistik ---")
print(df.describe().round(2))


Shape awal: (6347, 19)

--- Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6347 entries, 0 to 6346
Data columns (total 19 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   country_name                 6347 non-null   object 
 1   country_code                 6347 non-null   object 
 2   year                         6347 non-null   int64  
 3   region                       6347 non-null   object 
 4   income_group                 6347 non-null   object 
 5   gdp_growth_annual_pct        6347 non-null   float64
 6   gdp_growth_lag1              6347 non-null   float64
 7   gdp_growth_3yr_avg           6347 non-null   float64
 8   is_recession                 6347 non-null   int64  
 9   target_recession_next_year   6347 non-null   float64
 10  inflation_rate_pct           6347 non-null   float64
 11  unemployment_rate_pct        6347 non-null   float64
 12  lending_interest_rate_pct    6347 non-n

FIX TIPE DATA

In [ ]:
# target_recession_next_year masih float → ubah ke int
df['target_recession_next_year'] = df['target_recession_next_year'].astype(int)

# Pastikan year bertipe int
df['year'] = df['year'].astype(int)

# Kolom kategorikal → category (hemat memori)
cat_cols = ['country_name', 'country_code', 'region', 'income_group']
for col in cat_cols:
    df[col] = df[col].astype('category')

print("\n[OK] Tipe data diperbaiki")


[OK] Tipe data diperbaiki


# 5. TANGANI OUTLIER EKSTREM
# Strategi: WINSORIZE (clip) — lebih aman daripada hapus,
# karena ini data ekonomi yang memang bisa ekstrem

In [ ]:

def winsorize_col(series, lower_pct=0.01, upper_pct=0.99):
    """Clip nilai di luar percentile lower-upper."""
    lower = series.quantile(lower_pct)
    upper = series.quantile(upper_pct)
    return series.clip(lower, upper)

# Kolom yang perlu diwinsorisasi
cols_to_winsorize = [
    'gdp_growth_annual_pct',
    'gdp_growth_lag1',
    'gdp_growth_3yr_avg',
    'inflation_rate_pct',
    'lending_interest_rate_pct',
    'real_interest_rate_pct',
    'trade_pct_gdp',
    'external_debt_pct_gni',
    'reserves_months_import',
    'current_account_balance_usd',
]

df_clean = df.copy()
for col in cols_to_winsorize:
    df_clean[col] = winsorize_col(df[col])
    print(f"  Winsorize {col}: [{df[col].min():.2f}, {df[col].max():.2f}]"
          f" → [{df_clean[col].min():.2f}, {df_clean[col].max():.2f}]")

  Winsorize gdp_growth_annual_pct: [-54.40, 149.97] → [-17.02, 19.53]
  Winsorize gdp_growth_lag1: [-64.05, 149.97] → [-17.45, 19.53]
  Winsorize gdp_growth_3yr_avg: [-33.45, 80.11] → [-11.05, 15.79]
  Winsorize inflation_rate_pct: [-42.92, 225690.06] → [-10.94, 513.85]
  Winsorize lending_interest_rate_pct: [0.50, 1443.61] → [2.69, 57.70]
  Winsorize real_interest_rate_pct: [-4582.66, 228.09] → [-40.77, 37.53]
  Winsorize trade_pct_gdp: [0.02, 863.20] → [22.19, 322.71]
  Winsorize external_debt_pct_gni: [0.24, 1343.27] → [2.62, 251.27]
  Winsorize reserves_months_import: [0.00, 79.24] → [0.13, 21.52]
  Winsorize current_account_balance_usd: [-993142000000.00, 443374257116.43] → [-73909819795.85, 108639941348.83]


6. VALIDASI RANGE LOGIS

In [ ]:
# Unemployment tidak boleh negatif
df_clean['unemployment_rate_pct'] = df_clean['unemployment_rate_pct'].clip(lower=0)

# Population tidak boleh negatif
df_clean['population_total'] = df_clean['population_total'].clip(lower=0)

# is_recession & target hanya boleh 0 atau 1
assert df_clean['is_recession'].isin([0, 1]).all(), "is_recession ada nilai selain 0/1!"
assert df_clean['target_recession_next_year'].isin([0, 1]).all(), "target ada nilai selain 0/1!"

print("\n[OK] Validasi range logis selesai")


[OK] Validasi range logis selesai


7. SORT DATA (penting untuk analisis time series)

In [ ]:
df_clean = df_clean.sort_values(['country_code', 'year']).reset_index(drop=True)
print("[OK] Data diurutkan per negara per tahun")

[OK] Data diurutkan per negara per tahun


8. CEK KONSISTENSI LOGIS

In [ ]:
# Cek: is_recession=1 seharusnya gdp_growth negatif (tidak wajib, tapi perlu dicermati)
suspicious = df_clean[(df_clean['is_recession'] == 1) & (df_clean['gdp_growth_annual_pct'] > 5)]
print(f"\n[INFO] Baris resesi tapi GDP growth > 5%: {len(suspicious)} → perlu dicermati")
print(suspicious[['country_name', 'year', 'gdp_growth_annual_pct', 'is_recession']].head())


[INFO] Baris resesi tapi GDP growth > 5%: 0 → perlu dicermati
Empty DataFrame
Columns: [country_name, year, gdp_growth_annual_pct, is_recession]
Index: []


# 9. RINGKASAN AKHIR

In [ ]:
print("\n" + "="*50)
print("RINGKASAN CLEANING")
print("="*50)
print(f"Shape akhir  : {df_clean.shape}")
print(f"Missing values: {df_clean.isnull().sum().sum()}")
print(f"Duplikat     : {df_clean.duplicated().sum()}")
print(f"Tahun        : {df_clean['year'].min()} - {df_clean['year'].max()}")
print(f"Negara       : {df_clean['country_code'].nunique()}")
print(f"Distribusi target:")
print(df_clean['target_recession_next_year'].value_counts())


RINGKASAN CLEANING
Shape akhir  : (6347, 19)
Missing values: 0
Duplikat     : 0
Tahun        : 1992 - 2022
Negara       : 214
Distribusi target:
target_recession_next_year
0    5279
1    1068
Name: count, dtype: int64


Machine Learning (Pemrosesan Data)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# ── STYLE ────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
    'figure.dpi': 130,
})
BLUE   = '#185FA5'
RED    = '#E24B4A'
AMBER  = '#BA7517'
GREEN  = '#3B6D11'
PURPLE = '#534AB7'
TEAL   = '#0F6E56'
GRAY   = '#888780'
PALETTE = [BLUE, RED, AMBER, GREEN, PURPLE, TEAL, GRAY]

# ── LOAD & CLEAN ─────────────────────────────────────────────────────────────


df = pd.read_csv('/content/drive/MyDrive/data economic GDP/final_global_economic_crisis_dataset.csv')
df['target_recession_next_year'] = df['target_recession_next_year'].astype(int)



df_0 = df[df['target_recession_next_year'] == 0]
df_1 = df[df['target_recession_next_year'] == 1]

# ═══════════════════════════════════════════════════════════════════════════
# FIGURE 1 — OVERVIEW & CLASS BALANCE
# ═══════════════════════════════════════════════════════════════════════════
fig1, axes = plt.subplots(1, 3, figsize=(15, 4))
fig1.suptitle('EDA — Overview & Class Balance', fontsize=14, fontweight='bold', y=1.01)

# 1a. Target distribution
counts = df['target_recession_next_year'].value_counts()
bars = axes[0].bar(['Tidak Resesi\n(0)', 'Resesi\n(1)'], counts.values,
                    color=[BLUE, RED], width=0.5)
axes[0].set_title('Distribusi Target')
axes[0].set_ylabel('Jumlah Observasi')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 40,
                 f'{val:,}\n({val/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=10)
axes[0].set_ylim(0, counts.max() * 1.2)

# 1b. Recession rate per region
region_rate = df.groupby('region')['target_recession_next_year'].mean().sort_values()
colors_r = [RED if v > 0.18 else BLUE for v in region_rate.values]
axes[1].barh(region_rate.index, region_rate.values * 100, color=colors_r)
axes[1].set_title('Tingkat Resesi per Region (%)')
axes[1].set_xlabel('% tahun berikutnya resesi')
axes[1].axvline(region_rate.mean() * 100, color=AMBER, linestyle='--', linewidth=1, label='Rata-rata')
axes[1].legend(fontsize=9)
for i, (idx, val) in enumerate(region_rate.items()):
    axes[1].text(val * 100 + 0.3, i, f'{val*100:.1f}%', va='center', fontsize=9)

# 1c. Recession rate per income group
income_order = ['Low income', 'Lower middle income', 'Upper middle income', 'High income']
income_rate = df.groupby('income_group')['target_recession_next_year'].mean()
income_rate = income_rate.reindex(income_order)
axes[2].bar(range(4), income_rate.values * 100,
            color=[BLUE, TEAL, AMBER, RED], width=0.6)
axes[2].set_xticks(range(4))
axes[2].set_xticklabels(['Low', 'Lower mid', 'Upper mid', 'High'], fontsize=9)
axes[2].set_title('Tingkat Resesi per Income Group (%)')
axes[2].set_ylabel('%')
for i, val in enumerate(income_rate.values):
    axes[2].text(i, val * 100 + 0.3, f'{val*100:.1f}%', ha='center', fontsize=9)

fig1.tight_layout()
fig1.savefig('/content/drive/MyDrive/data ecommers/eda_01_overview.png', bbox_inches='tight')
plt.close()

# ═══════════════════════════════════════════════════════════════════════════
# FIGURE 2 — TIME SERIES: RECESSION RATE PER TAHUN
# ═══════════════════════════════════════════════════════════════════════════
fig2, axes = plt.subplots(2, 1, figsize=(14, 8))
fig2.suptitle('EDA — Tren Waktu', fontsize=14, fontweight='bold')

# 2a. Recession rate per year
yr = df.groupby('year')['target_recession_next_year'].mean() * 100
axes[0].fill_between(yr.index, yr.values, alpha=0.25, color=RED)
axes[0].plot(yr.index, yr.values, color=RED, linewidth=2)
axes[0].set_title('Tingkat Resesi Global per Tahun (%)')
axes[0].set_ylabel('% Negara akan Resesi')
axes[0].set_xlabel('')
crisis_years = {2008: 'GFC 2008', 1997: 'Asia\nCrisis', 2020: 'COVID-19', 2019: 'Pre-\nCOVID'}
for yr_mark, label in crisis_years.items():
    if yr_mark in yr.index:
        axes[0].axvline(yr_mark, color=AMBER, linestyle='--', linewidth=1, alpha=0.7)
        axes[0].text(yr_mark + 0.2, yr[yr_mark] + 2, label, fontsize=8, color=AMBER)

# 2b. GDP growth average per year (all countries)
gdp_yr = df.groupby('year')['gdp_growth_annual_pct'].mean()
axes[1].fill_between(gdp_yr.index, gdp_yr.values, 0,
                     where=gdp_yr.values >= 0, alpha=0.3, color=BLUE, label='Positif')
axes[1].fill_between(gdp_yr.index, gdp_yr.values, 0,
                     where=gdp_yr.values < 0, alpha=0.3, color=RED, label='Negatif')
axes[1].plot(gdp_yr.index, gdp_yr.values, color=BLUE, linewidth=1.5)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Rata-rata GDP Growth Global per Tahun (%)')
axes[1].set_ylabel('GDP Growth (%)')
axes[1].set_xlabel('Tahun')
axes[1].legend()

fig2.tight_layout()
fig2.savefig('/content/drive/MyDrive/data ecommers/eda_02_timeseries.png', bbox_inches='tight')
plt.close()

# ═══════════════════════════════════════════════════════════════════════════
# FIGURE 3 — DISTRIBUSI FITUR NUMERIK (Resesi vs Non-Resesi)
# ═══════════════════════════════════════════════════════════════════════════
num_features = [
    ('gdp_growth_annual_pct', 'GDP Growth (%)'),
    ('inflation_rate_pct', 'Inflasi (%)'),
    ('unemployment_rate_pct', 'Pengangguran (%)'),
    ('real_interest_rate_pct', 'Real Interest Rate (%)'),
    ('trade_pct_gdp', 'Trade % GDP'),
    ('external_debt_pct_gni', 'External Debt % GNI'),
    ('reserves_months_import', 'Cadangan Devisa (bulan)'),
    ('lending_interest_rate_pct', 'Lending Interest Rate (%)'),
]

fig3, axes = plt.subplots(2, 4, figsize=(16, 8))
fig3.suptitle('EDA — Distribusi Fitur: Resesi vs Non-Resesi', fontsize=14, fontweight='bold')
axes = axes.flatten()

for i, (col, label) in enumerate(num_features):
    # clip extreme for visualization
    clip_lo = df[col].quantile(0.02)
    clip_hi = df[col].quantile(0.98)
    data0 = df_0[col].clip(clip_lo, clip_hi)
    data1 = df_1[col].clip(clip_lo, clip_hi)

    axes[i].hist(data0, bins=35, color=BLUE, alpha=0.6, label='Tidak Resesi', density=True)
    axes[i].hist(data1, bins=35, color=RED, alpha=0.6, label='Resesi', density=True)
    axes[i].axvline(data0.mean(), color=BLUE, linestyle='--', linewidth=1.2)
    axes[i].axvline(data1.mean(), color=RED, linestyle='--', linewidth=1.2)
    axes[i].set_title(label, fontsize=10)
    axes[i].set_ylabel('Densitas' if i % 4 == 0 else '')
    if i == 0:
        axes[i].legend(fontsize=8)

fig3.tight_layout()
fig3.savefig('/content/drive/MyDrive/data ecommers/eda_03_distributions.png', bbox_inches='tight')
plt.close()

# ═══════════════════════════════════════════════════════════════════════════
# FIGURE 4 — KORELASI & HEATMAP
# ═══════════════════════════════════════════════════════════════════════════
fig4, axes = plt.subplots(1, 2, figsize=(16, 6))
fig4.suptitle('EDA — Korelasi', fontsize=14, fontweight='bold')

# 4a. Correlation with target (bar)
num_cols = df.select_dtypes(include='number').columns.tolist()
corr_target = df[num_cols].corr()['target_recession_next_year'].drop('target_recession_next_year').sort_values()
colors_c = [RED if v > 0 else BLUE for v in corr_target.values]
axes[0].barh(corr_target.index, corr_target.values, color=colors_c)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title('Korelasi Fitur dengan Target Resesi')
axes[0].set_xlabel('Korelasi Pearson')
for i, (idx, val) in enumerate(corr_target.items()):
    offset = 0.005 if val >= 0 else -0.005
    ha = 'left' if val >= 0 else 'right'
    axes[0].text(val + offset, i, f'{val:.3f}', va='center', ha=ha, fontsize=8)

# 4b. Correlation heatmap (numeric features only)
feat_cols = [c for c in num_cols if c not in ['year']]
corr_matrix = df[feat_cols].corr()
im = axes[1].imshow(corr_matrix.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
axes[1].set_xticks(range(len(feat_cols)))
axes[1].set_yticks(range(len(feat_cols)))
short_names = [c.replace('_pct', '').replace('_annual', '').replace('gdp_growth', 'gdp')
               .replace('_rate', '').replace('_total', '').replace('_months_import', '_res')
               .replace('current_account_balance_usd', 'cur_acc')
               .replace('external_debt_pct_gni', 'ext_debt') for c in feat_cols]
axes[1].set_xticklabels(short_names, rotation=45, ha='right', fontsize=8)
axes[1].set_yticklabels(short_names, fontsize=8)
for i in range(len(feat_cols)):
    for j in range(len(feat_cols)):
        val = corr_matrix.values[i, j]
        if abs(val) > 0.3:
            axes[1].text(j, i, f'{val:.2f}', ha='center', va='center',
                         fontsize=7, color='white' if abs(val) > 0.6 else 'black')
plt.colorbar(im, ax=axes[1], shrink=0.8)
axes[1].set_title('Heatmap Korelasi Antar Fitur')

fig4.tight_layout()
fig4.savefig('/content/drive/MyDrive/data ecommers/eda_04_correlation.png', bbox_inches='tight')
plt.close()

# ═══════════════════════════════════════════════════════════════════════════
# FIGURE 5 — BOXPLOT & SCATTER
# ═══════════════════════════════════════════════════════════════════════════
fig5, axes = plt.subplots(1, 3, figsize=(15, 5))
fig5.suptitle('EDA — Perbandingan Statistik & Scatter', fontsize=14, fontweight='bold')

# 5a. Boxplot GDP growth
data_box = [df_0['gdp_growth_annual_pct'].clip(-20, 20),
            df_1['gdp_growth_annual_pct'].clip(-20, 20)]
bp = axes[0].boxplot(data_box, patch_artist=True, widths=0.5,
                      medianprops=dict(color='white', linewidth=2))
bp['boxes'][0].set_facecolor(BLUE); bp['boxes'][0].set_alpha(0.7)
bp['boxes'][1].set_facecolor(RED);  bp['boxes'][1].set_alpha(0.7)
axes[0].set_xticklabels(['Tidak Resesi', 'Resesi'])
axes[0].set_title('GDP Growth vs Target')
axes[0].set_ylabel('GDP Growth (%)')
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')

# 5b. Boxplot inflation
inf_clip = df['inflation_rate_pct'].quantile(0.97)
data_inf = [df_0['inflation_rate_pct'].clip(-20, inf_clip),
            df_1['inflation_rate_pct'].clip(-20, inf_clip)]
bp2 = axes[1].boxplot(data_inf, patch_artist=True, widths=0.5,
                       medianprops=dict(color='white', linewidth=2))
bp2['boxes'][0].set_facecolor(BLUE); bp2['boxes'][0].set_alpha(0.7)
bp2['boxes'][1].set_facecolor(RED);  bp2['boxes'][1].set_alpha(0.7)
axes[1].set_xticklabels(['Tidak Resesi', 'Resesi'])
axes[1].set_title('Inflasi vs Target')
axes[1].set_ylabel('Inflasi (%)')

# 5c. Scatter GDP growth vs is_recession (color by target)
sample = df.sample(1000, random_state=42)
sc0 = sample[sample['target_recession_next_year'] == 0]
sc1 = sample[sample['target_recession_next_year'] == 1]
axes[2].scatter(sc0['gdp_growth_annual_pct'].clip(-25, 25),
                sc0['unemployment_rate_pct'].clip(0, 35),
                color=BLUE, alpha=0.4, s=15, label='Tidak Resesi')
axes[2].scatter(sc1['gdp_growth_annual_pct'].clip(-25, 25),
                sc1['unemployment_rate_pct'].clip(0, 35),
                color=RED, alpha=0.5, s=15, label='Resesi')
axes[2].set_xlabel('GDP Growth (%)')
axes[2].set_ylabel('Pengangguran (%)')
axes[2].set_title('GDP Growth vs Pengangguran')
axes[2].legend(fontsize=9)

fig5.tight_layout()
fig5.savefig('/content/drive/MyDrive/data ecommers/eda_05_boxplot_scatter.png', bbox_inches='tight')
plt.close()

print("Semua plot berhasil dibuat!")
print("Files: eda_01_overview.png, eda_02_timeseries.png,")
print("       eda_03_distributions.png, eda_04_correlation.png, eda_05_boxplot_scatter.png")


Semua plot berhasil dibuat!
Files: eda_01_overview.png, eda_02_timeseries.png,
       eda_03_distributions.png, eda_04_correlation.png, eda_05_boxplot_scatter.png


EDA (Exploratory Data Analysis)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# ── STYLE ────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
    'figure.dpi': 130,
})
BLUE   = '#185FA5'
RED    = '#E24B4A'
AMBER  = '#BA7517'
GREEN  = '#3B6D11'
PURPLE = '#534AB7'
TEAL   = '#0F6E56'
GRAY   = '#888780'
PALETTE = [BLUE, RED, AMBER, GREEN, PURPLE, TEAL, GRAY]

# ── LOAD & CLEAN ─────────────────────────────────────────────────────────────
df = pd.read_csv('/content/drive/MyDrive/data economic GDP/final_global_economic_crisis_dataset.csv')
df['target_recession_next_year'] = df['target_recession_next_year'].astype(int)
df_0 = df[df['target_recession_next_year'] == 0]
df_1 = df[df['target_recession_next_year'] == 1]

# ═══════════════════════════════════════════════════════════════════════════
# FIGURE 1 — OVERVIEW & CLASS BALANCE
# ═══════════════════════════════════════════════════════════════════════════
fig1, axes = plt.subplots(1, 3, figsize=(15, 4))
fig1.suptitle('EDA — Overview & Class Balance', fontsize=14, fontweight='bold', y=1.01)

# 1a. Target distribution
counts = df['target_recession_next_year'].value_counts()
bars = axes[0].bar(['Tidak Resesi\n(0)', 'Resesi\n(1)'], counts.values,
                    color=[BLUE, RED], width=0.5)
axes[0].set_title('Distribusi Target')
axes[0].set_ylabel('Jumlah Observasi')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 40,
                 f'{val:,}\n({val/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=10)
axes[0].set_ylim(0, counts.max() * 1.2)

# 1b. Recession rate per region
region_rate = df.groupby('region')['target_recession_next_year'].mean().sort_values()
colors_r = [RED if v > 0.18 else BLUE for v in region_rate.values]
axes[1].barh(region_rate.index, region_rate.values * 100, color=colors_r)
axes[1].set_title('Tingkat Resesi per Region (%)')
axes[1].set_xlabel('% tahun berikutnya resesi')
axes[1].axvline(region_rate.mean() * 100, color=AMBER, linestyle='--', linewidth=1, label='Rata-rata')
axes[1].legend(fontsize=9)
for i, (idx, val) in enumerate(region_rate.items()):
    axes[1].text(val * 100 + 0.3, i, f'{val*100:.1f}%', va='center', fontsize=9)

# 1c. Recession rate per income group
income_order = ['Low income', 'Lower middle income', 'Upper middle income', 'High income']
income_rate = df.groupby('income_group')['target_recession_next_year'].mean()
income_rate = income_rate.reindex(income_order)
axes[2].bar(range(4), income_rate.values * 100,
            color=[BLUE, TEAL, AMBER, RED], width=0.6)
axes[2].set_xticks(range(4))
axes[2].set_xticklabels(['Low', 'Lower mid', 'Upper mid', 'High'], fontsize=9)
axes[2].set_title('Tingkat Resesi per Income Group (%)')
axes[2].set_ylabel('%')
for i, val in enumerate(income_rate.values):
    axes[2].text(i, val * 100 + 0.3, f'{val*100:.1f}%', ha='center', fontsize=9)

fig1.tight_layout()
fig1.savefig('/content/drive/MyDrive/data ecommers/eda_01_overview.png', bbox_inches='tight')
plt.close()

# ═══════════════════════════════════════════════════════════════════════════
# FIGURE 2 — TIME SERIES: RECESSION RATE PER TAHUN
# ═══════════════════════════════════════════════════════════════════════════
fig2, axes = plt.subplots(2, 1, figsize=(14, 8))
fig2.suptitle('EDA — Tren Waktu', fontsize=14, fontweight='bold')

# 2a. Recession rate per year
yr = df.groupby('year')['target_recession_next_year'].mean() * 100
axes[0].fill_between(yr.index, yr.values, alpha=0.25, color=RED)
axes[0].plot(yr.index, yr.values, color=RED, linewidth=2)
axes[0].set_title('Tingkat Resesi Global per Tahun (%)')
axes[0].set_ylabel('% Negara akan Resesi')
axes[0].set_xlabel('')
crisis_years = {2008: 'GFC 2008', 1997: 'Asia\nCrisis', 2020: 'COVID-19', 2019: 'Pre-\nCOVID'}
for yr_mark, label in crisis_years.items():
    if yr_mark in yr.index:
        axes[0].axvline(yr_mark, color=AMBER, linestyle='--', linewidth=1, alpha=0.7)
        axes[0].text(yr_mark + 0.2, yr[yr_mark] + 2, label, fontsize=8, color=AMBER)

# 2b. GDP growth average per year (all countries)
gdp_yr = df.groupby('year')['gdp_growth_annual_pct'].mean()
axes[1].fill_between(gdp_yr.index, gdp_yr.values, 0,
                     where=gdp_yr.values >= 0, alpha=0.3, color=BLUE, label='Positif')
axes[1].fill_between(gdp_yr.index, gdp_yr.values, 0,
                     where=gdp_yr.values < 0, alpha=0.3, color=RED, label='Negatif')
axes[1].plot(gdp_yr.index, gdp_yr.values, color=BLUE, linewidth=1.5)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Rata-rata GDP Growth Global per Tahun (%)')
axes[1].set_ylabel('GDP Growth (%)')
axes[1].set_xlabel('Tahun')
axes[1].legend()

fig2.tight_layout()
fig2.savefig('/content/drive/MyDrive/data ecommers/eda_02_timeseries.png', bbox_inches='tight')
plt.close()

# ═══════════════════════════════════════════════════════════════════════════
# FIGURE 3 — DISTRIBUSI FITUR NUMERIK (Resesi vs Non-Resesi)
# ═══════════════════════════════════════════════════════════════════════════
num_features = [
    ('gdp_growth_annual_pct', 'GDP Growth (%)'),
    ('inflation_rate_pct', 'Inflasi (%)'),
    ('unemployment_rate_pct', 'Pengangguran (%)'),
    ('real_interest_rate_pct', 'Real Interest Rate (%)'),
    ('trade_pct_gdp', 'Trade % GDP'),
    ('external_debt_pct_gni', 'External Debt % GNI'),
    ('reserves_months_import', 'Cadangan Devisa (bulan)'),
    ('lending_interest_rate_pct', 'Lending Interest Rate (%)'),
]

fig3, axes = plt.subplots(2, 4, figsize=(16, 8))
fig3.suptitle('EDA — Distribusi Fitur: Resesi vs Non-Resesi', fontsize=14, fontweight='bold')
axes = axes.flatten()

for i, (col, label) in enumerate(num_features):
    # clip extreme for visualization
    clip_lo = df[col].quantile(0.02)
    clip_hi = df[col].quantile(0.98)
    data0 = df_0[col].clip(clip_lo, clip_hi)
    data1 = df_1[col].clip(clip_lo, clip_hi)

    axes[i].hist(data0, bins=35, color=BLUE, alpha=0.6, label='Tidak Resesi', density=True)
    axes[i].hist(data1, bins=35, color=RED, alpha=0.6, label='Resesi', density=True)
    axes[i].axvline(data0.mean(), color=BLUE, linestyle='--', linewidth=1.2)
    axes[i].axvline(data1.mean(), color=RED, linestyle='--', linewidth=1.2)
    axes[i].set_title(label, fontsize=10)
    axes[i].set_ylabel('Densitas' if i % 4 == 0 else '')
    if i == 0:
        axes[i].legend(fontsize=8)

fig3.tight_layout()
fig3.savefig('/content/drive/MyDrive/data ecommers/eda_03_distributions.png', bbox_inches='tight')
plt.close()

# ═══════════════════════════════════════════════════════════════════════════
# FIGURE 4 — KORELASI & HEATMAP
# ═══════════════════════════════════════════════════════════════════════════
fig4, axes = plt.subplots(1, 2, figsize=(16, 6))
fig4.suptitle('EDA — Korelasi', fontsize=14, fontweight='bold')

# 4a. Correlation with target (bar)
num_cols = df.select_dtypes(include='number').columns.tolist()
corr_target = df[num_cols].corr()['target_recession_next_year'].drop('target_recession_next_year').sort_values()
colors_c = [RED if v > 0 else BLUE for v in corr_target.values]
axes[0].barh(corr_target.index, corr_target.values, color=colors_c)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title('Korelasi Fitur dengan Target Resesi')
axes[0].set_xlabel('Korelasi Pearson')
for i, (idx, val) in enumerate(corr_target.items()):
    offset = 0.005 if val >= 0 else -0.005
    ha = 'left' if val >= 0 else 'right'
    axes[0].text(val + offset, i, f'{val:.3f}', va='center', ha=ha, fontsize=8)

# 4b. Correlation heatmap (numeric features only)
feat_cols = [c for c in num_cols if c not in ['year']]
corr_matrix = df[feat_cols].corr()
im = axes[1].imshow(corr_matrix.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
axes[1].set_xticks(range(len(feat_cols)))
axes[1].set_yticks(range(len(feat_cols)))
short_names = [c.replace('_pct', '').replace('_annual', '').replace('gdp_growth', 'gdp')
               .replace('_rate', '').replace('_total', '').replace('_months_import', '_res')
               .replace('current_account_balance_usd', 'cur_acc')
               .replace('external_debt_pct_gni', 'ext_debt') for c in feat_cols]
axes[1].set_xticklabels(short_names, rotation=45, ha='right', fontsize=8)
axes[1].set_yticklabels(short_names, fontsize=8)
for i in range(len(feat_cols)):
    for j in range(len(feat_cols)):
        val = corr_matrix.values[i, j]
        if abs(val) > 0.3:
            axes[1].text(j, i, f'{val:.2f}', ha='center', va='center',
                         fontsize=7, color='white' if abs(val) > 0.6 else 'black')
plt.colorbar(im, ax=axes[1], shrink=0.8)
axes[1].set_title('Heatmap Korelasi Antar Fitur')

fig4.tight_layout()
fig4.savefig('/content/drive/MyDrive/data ecommers/eda_04_correlation.png', bbox_inches='tight')
plt.close()

# ═══════════════════════════════════════════════════════════════════════════
# FIGURE 5 — BOXPLOT & SCATTER
# ═══════════════════════════════════════════════════════════════════════════
fig5, axes = plt.subplots(1, 3, figsize=(15, 5))
fig5.suptitle('EDA — Perbandingan Statistik & Scatter', fontsize=14, fontweight='bold')

# 5a. Boxplot GDP growth
data_box = [df_0['gdp_growth_annual_pct'].clip(-20, 20),
            df_1['gdp_growth_annual_pct'].clip(-20, 20)]
bp = axes[0].boxplot(data_box, patch_artist=True, widths=0.5,
                      medianprops=dict(color='white', linewidth=2))
bp['boxes'][0].set_facecolor(BLUE); bp['boxes'][0].set_alpha(0.7)
bp['boxes'][1].set_facecolor(RED);  bp['boxes'][1].set_alpha(0.7)
axes[0].set_xticklabels(['Tidak Resesi', 'Resesi'])
axes[0].set_title('GDP Growth vs Target')
axes[0].set_ylabel('GDP Growth (%)')
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')

# 5b. Boxplot inflation
inf_clip = df['inflation_rate_pct'].quantile(0.97)
data_inf = [df_0['inflation_rate_pct'].clip(-20, inf_clip),
            df_1['inflation_rate_pct'].clip(-20, inf_clip)]
bp2 = axes[1].boxplot(data_inf, patch_artist=True, widths=0.5,
                       medianprops=dict(color='white', linewidth=2))
bp2['boxes'][0].set_facecolor(BLUE); bp2['boxes'][0].set_alpha(0.7)
bp2['boxes'][1].set_facecolor(RED);  bp2['boxes'][1].set_alpha(0.7)
axes[1].set_xticklabels(['Tidak Resesi', 'Resesi'])
axes[1].set_title('Inflasi vs Target')
axes[1].set_ylabel('Inflasi (%)')

# 5c. Scatter GDP growth vs is_recession (color by target)
sample = df.sample(1000, random_state=42)
sc0 = sample[sample['target_recession_next_year'] == 0]
sc1 = sample[sample['target_recession_next_year'] == 1]
axes[2].scatter(sc0['gdp_growth_annual_pct'].clip(-25, 25),
                sc0['unemployment_rate_pct'].clip(0, 35),
                color=BLUE, alpha=0.4, s=15, label='Tidak Resesi')
axes[2].scatter(sc1['gdp_growth_annual_pct'].clip(-25, 25),
                sc1['unemployment_rate_pct'].clip(0, 35),
                color=RED, alpha=0.5, s=15, label='Resesi')
axes[2].set_xlabel('GDP Growth (%)')
axes[2].set_ylabel('Pengangguran (%)')
axes[2].set_title('GDP Growth vs Pengangguran')
axes[2].legend(fontsize=9)

fig5.tight_layout()
fig5.savefig('/content/drive/MyDrive/data ecommers/eda_05_boxplot_scatter.png', bbox_inches='tight')
plt.close()

print("Semua plot berhasil dibuat!")
print("Files: eda_01_overview.png, eda_02_timeseries.png,")
print("       eda_03_distributions.png, eda_04_correlation.png, eda_05_boxplot_scatter.png")


Semua plot berhasil dibuat!
Files: eda_01_overview.png, eda_02_timeseries.png,
       eda_03_distributions.png, eda_04_correlation.png, eda_05_boxplot_scatter.png
